In [2]:
data_path = "/Users/vibhas/Desktop/dl_git/Waste-Detection-Non-Bio-and-Bio-Project-Using-Yolov8/data.yaml"


In [3]:
pip install google

Note: you may need to restart the kernel to use updated packages.


# INSTALL ULTRALYTICS

In [4]:
!pip install ultralytics

# LOAD DATASET


In [5]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="zt49meyUQZK7ElhQ4HPh")
project = rf.workspace("phan-vn-khi").project("dataset-usskc")
version = project.version(1)
dataset = version.download("yolov8")
#!unzip -q /content/dataset-1/roboflow.zip -d /content/dataset-1/

  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 MB 4.5 MB/s  0:00:12m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 892.3/892.3 kB 9.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 5.2 MB/s  0:00:00 eta 0:00:01
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)
  Attempting uninstall: idna╺━━━━━━━━━━━━━━━━━━━ 4/8 [opencv-python-headless]
    Found existing installation: idna 3.11m━━━━━━━━━━━━━━━━━━━ 4/8 [opencv-python-headless]
    Uninstalling idna-3.11:m╺━━━━━━━━━━━━━━━━━━━ 4/8 [opencv-python-headless]
      Successfully uninstalled idna-3.1190m━━━━━━━━━━━━━━━━━━━ 4/8 [opencv-python-headless]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [roboflow] [opencv-python-headless]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorfl


Extracting Dataset Version Zip to dataset-1 in yolov8:: 100%|██████████| 18838/18838 [00:01<00:00, 12479.23it/s]


# TRAIN MODEL

In [7]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='dataset-1/data.yaml', epochs=20, batch=32)

Ultralytics 8.3.223 🚀 Python-3.11.13 torch-2.8.0 CPU (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=True, profil

KeyboardInterrupt: 

# TEST

In [ ]:
import os

# Duyệt thư mục /content/runs/detect
base_path = "/content/runs/detect"
for root, dirs, files in os.walk(base_path):
    if "best.pt" in files:
        print("Found best.pt at:", os.path.join(root, "best.pt"))

In [ ]:
from PIL import Image
best_model = YOLO('/content/runs/detect/train/weights/best.pt')
result = best_model.predict(source='/content/dataset-1/test/images/20211206_165056_jpg.rf.8550bdcfe7ac97c9100aebf1cd04b32f.jpg')
for r in result:
  print(r.boxes)
  im_array = r.plot()  # plot a BGR numpy array of predictions
  im = Image.fromarray(im_array[..., ::-1])  # RGB PIL image
  im.show()  # show image
  im.save('result.jpg')  # save image

# Show ảnh nhưng không được nên sẽ dùng matplotlib

In [ ]:
from matplotlib import pyplot as plt
best_model = YOLO('/content/runs/detect/train/weights/best.pt')
result = best_model.predict(source='/content/dataset-1/test/images/20211206_165056_jpg.rf.8550bdcfe7ac97c9100aebf1cd04b32f.jpg')
for r in result:
    print(r.boxes)  # In thông tin bounding boxes

    im_array = r.plot()  # BGR array với bounding box
    im_rgb = im_array[..., ::-1]  # Chuyển từ BGR -> RGB

    plt.imshow(im_rgb)
    plt.axis('off')
    plt.title("YOLOv8 Detection")
    plt.show()

    # Lưu ảnh nếu cần
    #Image.fromarray(im_rgb).save('result.jpg')

# ĐÁNH GIÁ KẾT QUẢ TRÊN TẬP TEST

In [ ]:
metrics = model.val(
    data='/content/dataset-1/data.yaml',  # YAML chứa đường dẫn test set
    split='test',                      # Đánh giá trên tập test
)

# Hiển thị kết quả đánh giá
print("\n📊 Evaluation Metrics:")
print(f"Precision (mean): {metrics.box.p.mean():.4f}") # tỉ lệ dự đoán đúng trong mỗi lần có đối tượng
print(f"Recall (mean): {metrics.box.r.mean():.4f}") # tỉ lệ phát hiện đối tượng thực sự có mặt
print(f"mAP@0.5: {metrics.box.ap50.mean():.4f}")
print(f"mAP@0.5:0.95: {metrics.box.ap.mean():.4f}")

In [ ]:
!cp /content/runs/detect/train/weights/best.pt "/content/drive/MyDrive/An Nguyen/"